In [ ]:
import pynucastro as pyna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#%matplotlib widget

In [ ]:
rates_zhou17_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Zhou2017/Empirical_beta_decay_zhou2017_R1')
rates_tian25_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Tian2025/Empirical_beta_decay_tian2025_R1')
rates_song21_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/alpha_empirical/NMHF2021/SONG_alpha_decay_R1')
rates_NMHF2021_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/alpha_empirical/NMHF2021/NMHF2021_alpha_decay_R1')


In [ ]:
print(len(rates_zhou17_reaclib.get_rates()),
      len(rates_tian25_reaclib.get_rates()),
      len(rates_song21_reaclib.get_rates()),
      len(rates_NMHF2021_reaclib.get_rates()))

Reaction rates from files

In [ ]:
rates_reaclib=pyna.rates.library.Library(libfile='Nuclear_Data/Original_files/Reaclib_18_9_20')
rates_reaclib_exp=pyna.rates.library.Library(libfile='Nuclear_Data/Just_experimental/Reaclib_exp')
rates_beta_reddi=pd.read_csv('Empirical_formulas/Beta_minus_empirical/Reddi2023/Empirical_beta_decay_reddi.csv')
sources_exp=list(pd.read_csv('Nuclear_Data/Just_experimental/sources_exp')['0'])
rates_viola=pd.read_csv('Empirical_formulas/Alpha_empirical/alpha_decays.dat',sep='   ',header=None,engine='python')
rates_zhou17=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Zhou2017/Empirical_beta_decay_zhou2017_R1')
rates_tian25=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Tian2025/Empirical_beta_decay_tian2025_R1')
rates_NMH2021=pyna.rates.library.Library(libfile='Empirical_formulas/Alpha_empirical/NMHF2021/NMHF2021_alpha_decay_R1')


rates_zhou17_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Zhou2017/Empirical_zhou2017_beta_decay_reaclib_R1')
rates_tian25_reaclib=pyna.rates.library.Library(libfile='Empirical_formulas/Beta_minus_empirical/Tian2025/Empirical_tian2025_beta_decay_reaclib_R1')

nuclei_in_the_reaction=pd.read_csv(r'Nuclear_Data/Original_files/winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])

In [ ]:
filter_alpha=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z+r.products[1].Z and r.Q>0)

filter_alpha_teo=pyna.RateFilter(
    products=['he4'],
    exact=False,
    max_reactants=1,
    max_products=2,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z+r.products[1].Z and r.Q>0 and r.source['Label'] not in sources_exp )

filter_beta_minus=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z-1 and r.Q>0)

filter_beta_minus_teo=pyna.RateFilter(
    max_reactants=1,
    max_products=1,
    filter_function=lambda r: r.reactants[0].Z==r.products[0].Z-1 and r.Q>0 and r.source['Label'] not in sources_exp)


rates_beta_reaclib=rates_reaclib.filter(filter_beta_minus_teo)
rates_alpha_reaclib=rates_reaclib.filter(filter_alpha_teo)
rates_beta_exp=rates_reaclib_exp.filter(filter_beta_minus)
rates_alpha_exp=rates_reaclib_exp.filter(filter_alpha)
rates_beta_reac_zhou=rates_zhou17_reaclib.filter(filter_beta_minus)



In [ ]:
print(len(rates_beta_reaclib.get_rates()),
      len(rates_alpha_reaclib.get_rates()),
        len(rates_beta_exp.get_rates()),
        len(rates_alpha_exp.get_rates()),)

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})

fig, ax = plt.subplots()

ax.set_xlabel("N")
ax.set_ylabel("Z")
ax.set_xlim(-0.5, 240.5)
ax.set_ylim(-0.5, 116.5)
ax.set_aspect("equal")
ax.set_title(r"$\beta$-decay")
fig.set_size_inches(14,10)

ax.scatter(nuclei_in_the_reaction['N'],
           nuclei_in_the_reaction['Z'],
           marker='s',color='gray',label='Nuclei in the network',alpha=0.1,s=5)

ax.scatter([r.reactants[0].N for r in rates_beta_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_beta_reaclib.get_rates()],
           marker="s", color="C1", label="Reaclib_Theoretical",s=2,alpha=0.5)

#ax.scatter([r.reactants[0].N for r in rates_beta_reac_zhou.get_rates()],
#           [r.reactants[0].Z for r in rates_beta_reac_zhou.get_rates()],
#           marker="s", color="C6", label="Zhou2017",s=2)

#ax.scatter(rates_beta_reddi['N'],
#           rates_beta_reddi['Z'],
#           marker="s", color="C3", label="Reddi",s=1)

ax.scatter([r.reactants[0].N for r in rates_tian25.get_rates()],
           [r.reactants[0].Z for r in rates_tian25.get_rates()],
           marker="s", color="C2", label="Tian2025",s=2)

ax.scatter([r.reactants[0].N for r in rates_beta_exp.get_rates()],
           [r.reactants[0].Z for r in rates_beta_exp.get_rates()],
           marker="s", color="C0", label="Experiments",s=1)

ax.legend()
fig.savefig("Beta_decay_Reaclib_Tian.pdf")


In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter([r.reactants[0].N for r in rates_beta_exp.get_rates()],
           [r.reactants[0].Z for r in rates_beta_exp.get_rates()],
           [np.log(r.eval(1e9)) for r in rates_beta_exp.get_rates()],
           label='Experimental beta',s=0.8,color='red')
ax.scatter([r.reactants[0].N for r in rates_beta_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_beta_reaclib.get_rates()],
           [np.log(r.eval(1e9)) for r in rates_beta_reaclib.get_rates()],
           label='reaclib beta',s=0.8,color='blue')
ax.scatter([r.reactants[0].N for r in rates_zhou17.get_rates()],
           [r.reactants[0].Z for r in rates_zhou17.get_rates()],
           [np.log(r.eval(1e9)) for r in rates_zhou17.get_rates()],
           label='reaclib beta',s=0.8,color='orange')


#ax.scatter(N_vioa,Z_vioa,np.log(lambda_vioa),label='Viola',s=1, marker='X',color='green')
plt.legend()
ax.set_zlabel('log(Decay rate (1/s))')
ax.set_zlim((-60,12))
plt.ylabel('Z of reactant')
plt.xlabel('N of reactant')
plt.show()

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12
})


fig, ax = plt.subplots()
ax.set_xlabel("N")
ax.set_ylabel("Z")
ax.set_xlim(-0.5, 240.5)
ax.set_ylim(-0.5, 116.5)
ax.set_aspect("equal")
ax.set_title(r"$\alpha$-decay")
fig.set_size_inches(14,10)

ax.scatter(nuclei_in_the_reaction['N'],
           nuclei_in_the_reaction['Z'],
           marker='s',color='gray',label='Nuclei in the network',alpha=0.1,s=5)
ax.scatter([r.reactants[0].N for r in rates_NMH2021.get_rates()],
           [r.reactants[0].Z for r in rates_NMH2021.get_rates()],
           marker='s',color="C4", label="rates_NMH2021",s=5)

ax.scatter([pyna.Nucleus(name).N for name in rates_viola[0]],
           [pyna.Nucleus(name).Z for name in rates_viola[0]],
           marker='x',color='C6', label='Viola seaborg',s=2,alpha=0.5)

ax.scatter([r.reactants[0].N for r in rates_alpha_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_alpha_reaclib.get_rates()],
           marker='s',color="C1", label="Theoretical_Reaclib",s=2,alpha=0.5)


ax.scatter([r.reactants[0].N for r in rates_alpha_exp.get_rates()],
           [r.reactants[0].Z for r in rates_alpha_exp.get_rates()],
           marker="x", color="C0", label="Experiments",s=2)



ax.legend()
fig.savefig('alpha_reaclib_viola-seaborg.pdf')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
'''
ax.scatter([pyna.Nucleus(name).N for name in rates_viola[0]],
           [pyna.Nucleus(name).Z for name in rates_viola[0]],
           [np.log(rate) for rate in rates_viola[1]],
           color='C1', label='Viola seaborg',s=0.5)

ax.scatter([r.reactants[0].N for r in rates_NMH2021.get_rates()],
           [r.reactants[0].Z for r in rates_NMH2021.get_rates()],
           [np.log(np.log(2)/r.eval(1e9)) for r in rates_NMH2021.get_rates()],
           color="C4", label="rates_NMH2021",s=5)
'''
ax.scatter([r.reactants[0].N for r in rates_alpha_exp.get_rates()],
           [r.reactants[0].Z for r in rates_alpha_exp.get_rates()],
           [np.log(np.log(2)/r.eval(1e9)) for r in rates_alpha_exp.get_rates()],
           color="C0", label="Alpha exp",s=0.5)

ax.scatter([r.reactants[0].N for r in rates_alpha_reaclib.get_rates()],
           [r.reactants[0].Z for r in rates_alpha_reaclib.get_rates()],
           [np.log(np.log(2)/r.eval(1e9)) for r in rates_alpha_reaclib.get_rates()],
           color="C2", label="Alpha reaclib",s=0.5)

plt.legend()
ax.set_zlabel('log(Decay rate (1/s))')
plt.ylabel('Z of reactant')
ax.set_zlim(-40,80)
plt.xlabel('N of reactant')
plt.show()